# Bài tập Chapter 3: Canny · Multi-scale Hough · RANSAC

Tự cài đặt bằng NumPy — **không** dùng `cv2.Canny`, Hough/RANSAC có sẵn ở phần cốt lõi.

Lý thuyết: [`Computer_Vision 24.pdf`](./Computer_Vision%2024.pdf)

# Thiết lập chung

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    cv2 = None

%matplotlib inline

plt.rcParams['figure.figsize'] = (6, 6)
plt.rcParams['image.cmap'] = 'gray'

In [ ]:
def show_image(img, title='', cmap='gray'):
    plt.figure(figsize=(5, 5))
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap)
    else:
        plt.imshow(img)
    plt.title(title)
    plt.axis('off')
    plt.show()


def show_images(images, titles=None, cols=3, cmap='gray', figsize=(14, 8)):
    n = len(images)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=figsize)
    for i, img in enumerate(images):
        plt.subplot(rows, cols, i + 1)
        if img.ndim == 2:
            plt.imshow(img, cmap=cmap)
        else:
            plt.imshow(img)
        if titles is not None:
            plt.title(titles[i])
        plt.axis('off')
    plt.tight_layout()
    plt.show()


def to_grayscale(img):
    """Chuyển ảnh sang grayscale float64 [0, 255]."""
    arr = np.asarray(img, dtype=np.float64)
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3 and arr.shape[2] >= 3:
        return 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
    raise ValueError('Anh dau vao phai la grayscale hoac RGB')

# Bài 1: Canny từ đầu

## Pipeline (PDF §1.4)

1. Gaussian blur (σ)
2. Sobel → Gx, Gy, M, θ
3. Non-maximum suppression *(Tuyền)*
4. Hysteresis + Otsu *(Trường)*

## Duy — Gaussian blur + Sobel (§1.4.1–1.4.2)

In [ ]:
def convolve2d_reflect(img, kernel):
    """Tích chập 2D, pad reflect — dùng cho Gaussian và Sobel."""
    img = np.asarray(img, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(img, ((ph, ph), (pw, pw)), mode='reflect')
    out = np.zeros_like(img, dtype=np.float64)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            patch = padded[i:i + kh, j:j + kw]
            out[i, j] = np.sum(patch * kernel)
    return out


def gaussian_kernel_2d(sigma):
    """Kernel Gaussian 2D chuẩn hoá tổng = 1 (PDF §1.4.1)."""
    if sigma <= 0:
        raise ValueError('sigma phai > 0')
    r = int(3 * sigma + 0.5)
    ax = np.arange(-r, r + 1, dtype=np.float64)
    xx, yy = np.meshgrid(ax, ax)
    kern = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    s = kern.sum()
    if s == 0:
        raise ValueError('kernel Gaussian khong hop le')
    return kern / s


def gaussian_blur(img, sigma=1.4):
    """
    Làm mờ Gaussian trước khi tính gradient (blur trước, Sobel sau).
    img: grayscale float hoặc ảnh màu (tự chuyển grayscale).
    """
    gray = to_grayscale(img)
    kernel = gaussian_kernel_2d(float(sigma))
    return convolve2d_reflect(gray, kernel)


def sobel_gradients(img):
    """
    Sobel trên ảnh đã blur. Trả về Gx, Gy, M, theta (rad).
    M = sqrt(Gx^2 + Gy^2) — không dùng |Gx| + |Gy| (PDF §1.3).
    theta vuông góc với cạnh; dùng arctan2(Gy, Gx).
    """
    gray = to_grayscale(img)
    kx = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]], dtype=np.float64)
    ky = kx.T
    gx = convolve2d_reflect(gray, kx)
    gy = convolve2d_reflect(gray, ky)
    magnitude = np.hypot(gx, gy)
    theta = np.arctan2(gy, gx)
    return gx, gy, magnitude, theta

In [ ]:
def make_test_edges(size=256):
    """Ảnh mức xám có hình vuông và đường chéo — kiểm thử gradient."""
    img = np.full((size, size), 40.0, dtype=np.float64)
    img[60:180, 60:180] = 200.0
    y, x = np.mgrid[0:size, 0:size]
    mask = (x + y) > size * 0.85
    img[mask] = 220.0
    rng = np.random.default_rng(0)
    img += rng.normal(0, 8, img.shape)
    return np.clip(img, 0, 255)


test_img = make_test_edges()
sigma = 1.4
blurred = gaussian_blur(test_img, sigma=sigma)
gx, gy, mag, theta = sobel_gradients(blurred)

show_images(
    [test_img, blurred, mag, np.abs(theta)],
    titles=['Ảnh gốc', f'Sau Gaussian (σ={sigma})', 'Magnitude M', '|θ| (rad)'],
    cols=4,
    figsize=(16, 4)
)

## Tuyền — Non-maximum Suppression (§1.4.3)

`# TODO: Sinh viên tự viết non_maximum_suppression`

In [ ]:
def non_maximum_suppression(magnitude, theta):
    raise NotImplementedError

## Trường — Hysteresis + Otsu + pipeline Canny (§1.4.4–1.6)

`# TODO: hysteresis_threshold`, `otsu_threshold`, `canny_from_scratch`

In [ ]:
def hysteresis_threshold(magnitude, low, high):
    raise NotImplementedError


def otsu_threshold(magnitude):
    raise NotImplementedError


def canny_from_scratch(img, sigma=1.4, low_ratio=0.5, use_otsu=True):
    raise NotImplementedError

# Bài 2: Multi-scale Hough *(Nguyên, Nhung)*

PDF §2 — placeholder cho các hàm Hough.

In [ ]:
def hough_accumulator(edges, theta_res=1.0, rho_res=1.0):
    h, w = edges.shape
    diag_len = int(np.ceil(np.sqrt(h**2 + w**2)))
    rhos = np.arange(-diag_len, diag_len + rho_res, rho_res)
    thetas = np.deg2rad(np.arange(0, 180, theta_res))
    accumulator = np.zeros((len(rhos), len(thetas)), dtype=np.int32)

    ys, xs = np.nonzero(edges)

    for x, y in zip(xs, ys):
        for theta_idx, theta in enumerate(thetas):
            rho = x * np.cos(theta) + y * np.sin(theta)
            rho_idx = np.argmin(np.abs(rhos - rho))
            accumulator[rho_idx, theta_idx] += 1

    return accumulator, rhos, thetas


def find_top_k_peaks(accumulator, k=5, nms_radius=10):
    raise NotImplementedError


def gradient_directed_hough_voting(edges, gradient_theta, window_deg=20.0):
    raise NotImplementedError

## Demo Multi-Scale

In [ ]:
coarse_acc, coarse_rhos, coarse_thetas = hough_accumulator(edges, theta_res=4, rho_res=8)
medium_acc, medium_rhos, medium_thetas = hough_accumulator(edges, theta_res=1, rho_res=2)
fine_acc, fine_rhos, fine_thetas = hough_accumulator(edges, theta_res=0.25, rho_res=0.5)

## So sánh Accumulator

In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(131)
plt.imshow(coarse_acc, cmap="hot", aspect="auto")
plt.title("Coarse")

plt.subplot(132)
plt.imshow(medium_acc, cmap="hot", aspect="auto")
plt.title("Medium")

plt.subplot(133)
plt.imshow(fine_acc, cmap="hot", aspect="auto")
plt.title("Fine")

plt.tight_layout()
plt.show()

## Peak của từng scale

In [ ]:
print("Coarse peak:", np.max(coarse_acc))
print("Medium peak:", np.max(medium_acc))
print("Fine peak:", np.max(fine_acc))

## Multi-scale Hough

Thử nghiệm Hough Transform ở ba mức phân giải khác nhau:

- Coarse: Δρ = 8 px, Δθ = 4°
- Medium: Δρ = 2 px, Δθ = 1°
- Fine: Δρ = 0.5 px, Δθ = 0.25°

Mức coarse có tốc độ xử lý nhanh hơn nhưng độ chính xác thấp hơn. Mức fine cho kết quả chính xác hơn nhưng chi phí tính toán lớn hơn. Đây là cơ sở cho chiến lược coarse-to-fine trong Multi-scale Hough Transform.

# Bài 3: RANSAC *(Quyến)*

PDF §3 — placeholder.

In [ ]:
def ransac(points, fit_fn, distance_fn, s, epsilon, n_iterations, p_success=0.99):
    raise NotImplementedError

# Checklist trước khi nộp

- [ ] Bài 1: Canny đủ 4 bước + Otsu + so sánh IoU
- [ ] Bài 2: Multi-scale Hough + top-k + gradient voting
- [ ] Bài 3: RANSAC 2 hình dạng s < 5 + giải thích
- [ ] Không dùng OpenCV ở phần cốt lõi